# ML-10 — Action Playbook and Ranked Queue

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sohila-Khaled-Abbas/flyrank-ml-internship-starter/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

Generating the final ranked content refresh recommendation playbook with human-readable reason codes, intended use bounds, human-review rules, monitoring retrain triggers, and export files.

## 1. Ranked actions + reason codes

*Mapping top priority scores to human-readable reason codes and explicit editorial action labels.*

In [1]:
import pandas as pd
import numpy as np
import os
from sklearn.ensemble import RandomForestClassifier

csv_path = "../../data/raw/content_refresh_anonymized.csv"
if not os.path.exists(csv_path):
    csv_path = "data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(csv_path)
df["pos_clean"] = np.where(df["avg_position"] == 0, 50.0, df["avg_position"])
df["click_drop"] = np.maximum(0, df["clicks_prev_30d"] - df["clicks_last_30d"])
df["target_binary"] = (df["click_drop"] > df["click_drop"].median()).astype(int)

features = ["impressions_90d", "clicks_prev_30d", "pos_clean", "content_age_days", "ctr"]
rf = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42)
rf.fit(df[features], df["target_binary"])

df["model_score"] = rf.predict_proba(df[features])[:, 1]

conditions = [
    (df["click_drop"] > 10) & (df["impressions_90d"] > 1000),
    (df["content_age_days"] > 180) & (df["click_drop"] > 0),
    (df["avg_position"] >= 4) & (df["avg_position"] <= 10) & (df["ctr"] < 1.0)
]
reason_codes = ["STEEP_DROP_HIGH_VOLUME", "STALE_CONTENT_DECAY", "CTR_POSITION_UNDERPERFORMER"]
action_labels = ["REFRESH_AND_EXPAND_CONTENT", "UPDATE_STATISTICS_AND_FRESHNESS", "OPTIMIZE_TITLE_AND_META_TAGS"]

df["reason_code"] = np.select(conditions, reason_codes, default="MONITOR_PERFORMANCE")
df["action_label"] = np.select(conditions, action_labels, default="NO_ACTION_NEEDED")

ranked_playbook = df.sort_values(by="model_score", ascending=False)
print("=== TOP 10 RANKED ACTION PLAYBOOK QUEUE ===")
print(ranked_playbook[["content_id", "client_id", "model_score", "click_drop", "reason_code", "action_label"]].head(10).to_string(index=False))


=== TOP 10 RANKED ACTION PLAYBOOK QUEUE ===
          content_id         client_id  model_score  click_drop         reason_code                    action_label
content_920bb86da524 client_19581e27de     0.986273           1 STALE_CONTENT_DECAY UPDATE_STATISTICS_AND_FRESHNESS
content_bc21baf71fb4 client_19581e27de     0.983512           1 STALE_CONTENT_DECAY UPDATE_STATISTICS_AND_FRESHNESS
content_2557c1c418b6 client_19581e27de     0.982612           1 STALE_CONTENT_DECAY UPDATE_STATISTICS_AND_FRESHNESS
content_4897e6088934 client_19581e27de     0.979967           1 STALE_CONTENT_DECAY UPDATE_STATISTICS_AND_FRESHNESS
content_910077c1bba4 client_19581e27de     0.977430           1 STALE_CONTENT_DECAY UPDATE_STATISTICS_AND_FRESHNESS
content_0e8a62788a4a client_19581e27de     0.977297           1 STALE_CONTENT_DECAY UPDATE_STATISTICS_AND_FRESHNESS
content_2105b32488af client_9f14025af0     0.977153           1 STALE_CONTENT_DECAY UPDATE_STATISTICS_AND_FRESHNESS
content_a760753ae6f1 client_

## 2. Intended use and limits

### **Intended Use**
- Decision-support priority scoring to guide content editorial teams toward high-ROI refresh candidates.

### **Known Limits**
- **Non-Causal**: Model ranks observed traffic decay opportunity; it does not guarantee search engine rank restoration.
- **Snapshot Data**: Evaluation is based on a 90-day static window snapshot; seasonal traffic variations require human context.
- **Unranked Proxy**: Pages with `avg_position = 0` are assigned a default proxy position of `50.0`.

## 3. Human review + the no-go list

### **Human Review Rules**
1. Editors must read the target URL before making updates to confirm that current search intent matches the refresh topic.
2. Verify that traffic decline is not caused by permanent brand query shifts or product deprecation.

### **No-Go List (What NOT to Automate)**
- 🚫 **Auto-Publishing AI Content**: Never automatically publish AI-generated text directly to live CMS without human editor review.
- 🚫 **Automated 301 Redirects**: Never auto-redirect or delete URLs based solely on low model priority scores.
- 🚫 **Automated Site-wide Title Changes**: Never run bulk automated meta updates across entire client domains without human verification.

## 4. Monitoring / retrain triggers

- **Monitoring**: Track post-refresh CTR and click recovery over a 30-day post-update window.
- **Retrain Triggers**:
  1. *Precision Drop*: Retrain model if Precision@50 on weekly holdout data drops below 85%.
  2. *Domain Distribution Shift*: Retrain model when new client domain profiles (e.g. e-commerce vs SaaS) are onboarded.

## 5. Exports for the paper

*Exporting the ranked action queue CSV to `work/outputs/action_playbook_queue.csv` and metrics receipt.*

In [2]:
output_dir = "../../work/outputs"
if not os.path.exists(output_dir):
    output_dir = "work/outputs"
os.makedirs(output_dir, exist_ok=True)

export_cols = ["content_id", "client_id", "model_score", "reason_code", "action_label", "impressions_90d", "clicks_90d", "content_age_days", "click_drop"]
export_path = os.path.join(output_dir, "action_playbook_queue.csv")
ranked_playbook[export_cols].to_csv(export_path, index=False)
print(f"Successfully exported action playbook queue ({len(ranked_playbook):,} rows) to: {export_path}")


Successfully exported action playbook queue (30,000 rows) to: ../../work/outputs\action_playbook_queue.csv


## Self-check

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/`